# Distancias y similitudes
En la sección previa convertimos un problema en una matriz $X \in \mathbb{R}^{n\times p}$. Aquí
respondemos la siguiente pregunta: **dados dos renglones de $X$, ¿cómo medimos qué tan parecidos
son?**

Esto no es un desvío teórico. Ya que muchoa algoritmos, k-centers que implementaremos mas adelante consiste, esencialmente, en repetir dos operaciones sobre distancias:

```
para cada punto:  asignarlo al centro mas CERCANO      -> argmin de distancias
nuevo centro:     el punto mas LEJANO a todos los centros -> argmax de distancias
```

Todo el algoritmo cabe en esas dos líneas. Lo que decide si funciona o no es **la función de
distancia** que uses y **la escala** de tus características.


## Preparación

Importamos las funciones de distancia de **baile**, la librería del curso. Son las mismas que
usará tu implementación de k-centers, así que conviene familiarizarse con ellas desde ahora.


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets

# baile: la libreria del curso
src = os.path.join(os.getcwd(), 'src')
if src not in sys.path:
    sys.path.append(src)
from SimpleClustering import euclidiana, coseno

iris = datasets.load_iris()
X = iris.data
Y = iris.target
print('X:', X.shape)

X: (150, 4)


In [ ]:
# Asi estan implementadas en src/SimpleClustering.py
import inspect
print(inspect.getsource(euclidiana))
print(inspect.getsource(coseno))

## 1. Estableciendo relaciones entre datos

Las matemáticas nos permiten establecer relaciones entre los números reales; las más utilizadas
son del tipo $>,<,\geq, \leq, =, \neq$. En análisis de datos, establecer relaciones entre
elementos de un conjunto de datos es de capital importancia.

Las escalas de medición que vimos  (nominal, ordinal, intervalo, razón) deben
tenerse en cuenta, porque ciertas operaciones matemáticas son apropiadas solo para algunos tipos
específicos. Cuando tratamos con datos numéricos, las relaciones se establecen a menudo mediante
medidas de **disimilitud/distancia** (Lebesgue, Minkowski, Bray-Curtis, Canberra) o mediante
medidas de **similitud** (coseno, Dice, Jaccard, Tanimoto). Las secuencias se pueden analizar
usando relaciones de secuencia (distancia de Hamming o distancia de edición).


## 2. La matriz de relaciones $R$

Dado un conjunto de elementos (abstractos), sin referirse necesariamente  a vectores de características numéricas.


$$O=\{o_1,o_2,\dots,o_n\}$$


A veces, no hay una representación de vector de características disponible para los objetos $o_k, k = 1, \dots , n$, por lo que los métodos convencionales de análisis de datos basados en características no son aplicables (al menos no directamente). En cambio, la relación de todos los pares de objetos a menudo se puede cuantificar y escribir como una matriz cuadrada.


$$R=\left(
  \begin{array}{cccc}
  r_{11} & r_{12}&\dots&r_{1n}\\
  r_{21} & r_{22}&\dots&r_{2n}\\
  \vdots & \vdots & \dots & \vdots \\
  r_{n1} & r_{n2}&\dots&r_{nn}\\
  \end{array}
\right) \in \mathbb{R}^{ n \times n} $$


donde cada valor de relación $r_{ij} , i, j = 1, \dots, n$ en $R$, puede referirse a un grado de similitud, disimilitud, compatibilidad, incompatibilidad, proximidad o distancia entre el par de objetos $o_i$ y $o_j$. Casi siempre se pide que la relación sea simétrica, es decir $r_{ij}=r_{ji}$ para todo $i,j = 1, \dots,n$. La matriz de relaciones $R$ puede definirse manualmente o calcularse a partir de las características. Si las características numéricas $X$ están disponibles, entonces $R$ puede calcularse a partir de $X$ usando una función apropiada $f : \mathbb{R}^p  \times \mathbb{R}^p  \rightarrow R$. Por ejemplo, una función relacional para las matriz para Iris puede ser definida manualmente por un botánico que compare visualmente y luego asigne numéricamente algunas relaciones entre pares de flores, o bien $R$ puede calcularse a partir de las longitudes y anchos de sépalos y pétalos.  A continuación presentaremos algunas relaciones de similitud y distancia.





Fíjate en el tamaño de $R$: es $n \times n$. Para las 150 flores de Iris son 22 500 valores,
y para un conjunto de un millón de elementos serían $10^{12}$. **Este es el motivo por el que
k-means existe**: k-centers y los métodos basados en la matriz completa no escalan, mientras que
k-means solo necesita las distancias de cada punto a $k$ centros, es decir $n \times k$ valores.
Lo veremos con números al final del curso.


## 3. Medidas de disimilitud (distancias)

Una función $d$ se llama disimilitud o medida de distancia si para todo $x, y \in \mathbb{R}^p$.

$$\begin{array}{l}
d(x,y)=d(y,x)\\
d(x,y)=0 \Leftrightarrow x=y\\
d(x,z)\leq d(x,y)+d(y,z)
\end{array}$$

de los axiomas previos se sigue que $d(x,y) \geq 0$. Una clase de medidas de disimilitud se define mediante la norma $\lVert . \rVert$ de $x - y$, por lo que

$d(x,y)=\lVert x-y \rVert$

Una función $\lVert . \rVert : \mathbb{R}^p \rightarrow \mathbb{R}$ es una norma si y solo si

$$\begin{array}{l}
\lVert x \rVert = 0 \Leftrightarrow x=(0,\dots,0) \\
\lVert a.x \rVert = |a| \cdot \lVert x \rVert  ~\forall_a\in\mathbb{R}, x \in \mathbb{R}^p\\
\lVert x+y \rVert \leq \lVert x \rVert+\lVert y \rVert ~\forall_{x,y} \in \mathbb{R}^p
\end{array}$$

Por ejemplo, la llamada norma hiperbólica, dada por la siguiente ecuación

$$
\lVert x \rVert_h = \Pi_{i=1}^p x^{(i)}
$$

no es una norma ya que, por ejemplo, la condición $\lVert x \rVert = 0 \Leftrightarrow x=(0,\dots,0)$ no se mantiene cuando $x=(0,1)$  ya que $\lVert x \rVert_h=0$ a pesar de que $x \neq (0,0)$.

Las clases de normas utilizadas con frecuencia son las normas matriciales y las normas de Lebesgue o Minkowski. La norma matricial se define como

$$\lVert x \rVert_A=\sqrt{x A x^T}$$

con una matriz $A \in \mathbb{R}^{p \times p}$. Algunos casos importantes para la matriz $A$ se definen en la siguiente tabla.





> **Ejercicio 3.1** La tercera propiedad de la lista, $\lVert x+y \rVert \leq \lVert x \rVert +
> \lVert y \rVert$, es la desigualdad del triángulo. Da un ejemplo concreto con $x,y\in\mathbb{R}^2$
> donde la desigualdad sea **estricta**, y otro donde se cumpla la igualdad. ¿Qué tienen de especial
> los vectores del segundo caso?


|Nombre|$A$|Comentario|
|----|---|--|
|Euclidiana| $$\left(
  \begin{array}{cccc}
  1 & 0 &\dots&0\\
  0 & 1 &\dots& 0\\
  \vdots & \vdots & \ddots & \vdots \\
  0 &  0 &\dots & 1\\
  \end{array}
\right)$$ ||
|Frobenius o Hilbert-Schmidt| $$\left(
  \begin{array}{cccc}
  1 & 1 &\dots& 1\\
  1 & 1 &\dots& 1\\
  \vdots & \vdots & \ddots & \vdots \\
  1 &  1 &\dots & 1\\
  \end{array}
\right)$$ ||
|diagonal | $$\left(
  \begin{array}{cccc}
  d_1 & 0 &\dots&0\\
  0 & d_2 &\dots& 0\\
  \vdots & \vdots & \ddots & \vdots \\
  0 &  0 &\dots & d_p\\
  \end{array}
\right)$$ | cada atributo $i$ es ponderado por $d_i$ |
|Mahalanobis | $\textit{cov}^{-1}X$ |Adapta la ponderación de las características individuales en función de las estadísticas observadas.|

### 3.1 La familia de normas de Lebesgue o Minkowski

La norma de Lebesgue o Minkowski se define como

$$
\lVert x \rVert_{\alpha} = \sqrt[\alpha]{\sum_{j=1}^p |x^{(j)}|^{\alpha}}
$$

con $\alpha \in  \mathbb{R} \setminus \{0\}$, que es igual a la media generalizada  excepto por un factor constante $\sqrt[\alpha]{n}$. Casos especiales importantes de la norma de Lebesgue o Minkowski se resumen en la siguiente tabla.

|Nombre|definición||
|------|---------|-------|
|Ínfimo $\alpha \rightarrow -\infty$| $$\lVert x \rVert_{-\infty}= \min_{j=1,2\dots p} |x^{(j)}| $$ | |
|Manhattan (city block) $\alpha=1$|$$
\lVert x \rVert_{\alpha} = \sum_{j=1}^p |x^{(j)}|
$$ | |
|Euclidiana $\alpha=2$|$$
\lVert x \rVert_{2} = \sqrt{\sum_{j=1}^p (x^{(j)})^{2}}
$$||
|Supremo $\alpha \rightarrow \infty$| $$\lVert x \rVert_{\infty}= \max_{j=1,2\dots p} |x^{(j)}| $$ | |

### 3.2 Implementación

Una sola función cubre toda la familia. Compárala con `euclidiana` de baile para el caso
$\alpha=2$.


In [ ]:
def minkowski(x, y, alpha=2):
    """Distancia de Minkowski de orden alpha entre dos vectores."""
    return np.sum(np.abs(x - y) ** alpha) ** (1.0 / alpha)

a, b = X[0], X[1]   # dos flores de la misma especie
c    = X[100]       # una flor de otra especie

print('flor 0  :', a)
print('flor 1  :', b)
print('flor 100:', c)
print()
print('Manhattan  (alpha=1)   d(0,1) = %.3f   d(0,100) = %.3f' % (minkowski(a,b,1), minkowski(a,c,1)))
print('Euclidiana (alpha=2)   d(0,1) = %.3f   d(0,100) = %.3f' % (minkowski(a,b,2), minkowski(a,c,2)))
print('Supremo    (alpha->inf) d(0,1) = %.3f   d(0,100) = %.3f'
      % (np.max(np.abs(a-b)), np.max(np.abs(a-c))))
print()
print('euclidiana de baile:  d(0,1) = %.3f   d(0,100) = %.3f' % (euclidiana(a,b), euclidiana(a,c)))

Las tres distancias coinciden en lo importante: la flor 0 está más cerca de la flor 1 (su
misma especie) que de la flor 100. **Esa es la apuesta de fondo de todo el aprendizaje basado en
distancias**: que los objetos parecidos según la métrica sean también parecidos según el problema.
Cuando esa apuesta falla, ningún algoritmo lo arregla.


## 4. La matriz de distancias de Iris

Hasta aquí $R$ ha sido una definición. Vamos a construirla.

La implementación es deliberadamente ingenua: dos ciclos anidados, igual que la definición. Para
$n=150$ son 22 500 evaluaciones, que corren en menos de un segundo.


In [ ]:
def matriz_distancias(datos, distancia=euclidiana):
    """Matriz R de n x n con la distancia entre todos los pares de elementos."""
    n = len(datos)
    R = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            R[i, j] = distancia(datos[i], datos[j])
    return R

R = matriz_distancias(X)
print('R:', R.shape)
print()
print('Comprobamos los axiomas sobre los datos reales:')
print('  diagonal cero d(x,x)=0 :', np.allclose(np.diag(R), 0))
print('  simetrica d(x,y)=d(y,x):', np.allclose(R, R.T))
print('  no negativa            :', (R >= 0).all())

In [ ]:
# Los datos vienen ordenados por especie, asi que la estructura se ve directamente
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(R, cmap='viridis')
for corte in [50, 100]:
    ax.axhline(corte - 0.5, color='white', lw=1)
    ax.axvline(corte - 0.5, color='white', lw=1)
ax.set_xticks([25, 75, 125]); ax.set_xticklabels(iris.target_names)
ax.set_yticks([25, 75, 125]); ax.set_yticklabels(iris.target_names)
ax.set_title('Matriz de distancias euclidianas de Iris')
fig.colorbar(im, label='distancia')
plt.show()

Esta figura es el resultado más importante del notebook. Léela con cuidado:

- El **bloque oscuro de arriba a la izquierda** son las 50 setosa: distancias pequeñas entre
  ellas, es decir, un grupo compacto.
- Ese mismo bloque es **claro** contra las otras dos especies: setosa está lejos de todo lo demás.
- Los bloques de versicolor y virginica son oscuros entre sí: esas dos especies **se traslapan**.

Nadie le dijo a `matriz_distancias` cuáles eran las especies. La estructura de bloques emergió
sola de las medidas de los pétalos y los sépalos. **Eso es exactamente lo que un algoritmo de
agrupamiento va a explotar**, y también anticipa el resultado: separar setosa será fácil,
separar versicolor de virginica no.


In [ ]:
# Distancia promedio dentro de cada especie y entre especies
nombres = iris.target_names
tabla = pd.DataFrame(index=nombres, columns=nombres, dtype=float)
for i, a in enumerate(nombres):
    for j, b in enumerate(nombres):
        tabla.loc[a, b] = R[np.ix_(Y == i, Y == j)].mean()
print('Distancia euclidiana promedio entre grupos:')
tabla

> **Ejercicio 4.1** La diagonal de esta tabla mide qué tan *compacta* es cada especie y el
> resto qué tan *separadas* están. ¿Qué par de especies será más difícil de distinguir? Contrasta
> tu respuesta con la gráfica de dispersión del notebook 01.


### 3.3 Distancia de Hamming

Otra disimilitud frecuentemente utilizada es la distancia de Hamming definida como:

 $$d_H(x,y)= \sum_{i=1}^p \rho(x^{(i)},y^{(i)}) $$

 donde

 $\rho(x,y)=\begin{cases}
 0 \text{ si } x=y\\
 1 \text{ en cualquier otro caso}
 \end{cases}$

 Note que la distancia de Hamming cuenta el número de valores de características que no coinciden. Para características binarias, la distancia de Hamming es igual a la distancia de Manhattan, $d_H(x,y) = \lVert x-y \rVert_1$. Observe, sin embargo, que la distancia de Hamming no está asociada con una norma porque la condición $\lVert a.x \rVert = |a| \cdot \lVert x \rVert  \forall_a\in\mathbb{R}, x \in \mathbb{R}^p$ no se cumple. Las variantes de la distancia de Hamming usan funciones modificadas $\rho$ para especificar similitudes entre características individuales. Por ejemplo, si las características son (escala nominal) documentos de texto, entonces $\rho$ podría ser menor para pares  con contenido similar y mayor para pares de documentos con contenido muy diferente.

Aquí se cierra el círculo con el notebook 01: la codificación **one-hot** convierte una
característica nominal en un vector binario, y sobre vectores binarios la distancia de Manhattan
*es* la distancia de Hamming, que solo cuenta en cuántas posiciones difieren. Por eso one-hot
respeta la naturaleza de una característica nominal y label encoding no.


## 5. Medidas de similitud

Una función s se llama medida de similitud o proximidad si para todo $x, y \in \mathbb{R}^p$

$$\begin{array}{l}
s(x,y)=s(y,x)\\
s(x,x)\geq s(x,y)\\
s(x,y)\geq 0
\end{array}$$

Adicionalmente, si $s(x,y) \in [0,1]$ para todo $x,y$ (equivalentemente, si $s(x,x)=1$), se dice que es una función de similitud **normalizada**.
Cualquier medida de disimilitud $d$ se puede usar para definir una medida de similitud  $s$ y viceversa, por ejemplo, usando una función positiva monótonamente decreciente $f$ con $f (0) = 1$ como

$$s(x,y)=\frac{1}{1+d(x,y)}$$

Consideremos primero las similitudes entre los vectores de características binarias. Un par de vectores de características binarias pueden ser considerados similares si muchos de sus 1's coinciden. Esta conjunción puede ser representada por el producto, por lo que el producto escalar de los vectores de características es un candidato razonable para una medida de similitud. También para características de valor real no negativo $x,y \in (\mathbb{R^+})^p$ las medidas de similitud se pueden definir en base a productos escalares que se pueden normalizar de diferentes maneras:

- Similitud coseno

$$s(x,y)=\frac{\sum_{i=1}^p x^{(i)}y^{(i)}}{\sqrt{\sum_{i=1}^p (x^{(i)})^2 \sum_{i=1}^p (y^{(i)})^2}}$$

- Similitud traslape

$$s(x,y)=\frac{\sum_{i=1}^p x^{(i)}y^{(i)}}{\min\left(\sum_{i=1}^p (x^{(i)})^2 ,\sum_{i=1}^p (y^{(i)})^2\right)}$$


- Similitud Dice (dado)

$$s(x,y)=\frac{2\sum_{i=1}^p x^{(i)}y^{(i)}}{\sum_{i=1}^p (x^{(i)})^2 +\sum_{i=1}^p (y^{(i)})^2}$$

- Jaccard (Tanimoto)


$$s(x,y)=\frac{\sum_{i=1}^p x^{(i)}y^{(i)}}{\sum_{i=1}^p (x^{(i)})^2 +\sum_{i=1}^p (y^{(i)})^2- \sum_{i=1}^p x^{(i)}y^{(i)}}$$




Estas expresiones no están definidas para vectores en el que todas características son cero porque los denominadores son cero, por lo que la similitud debe definirse explícitamente para este caso, por ejemplo, como cero.

La similitud coseno mide el **ángulo** entre dos vectores, ignorando su magnitud. La función
`coseno` de baile devuelve $1 - s(x,y)$, es decir, la *disimilitud* asociada, para poder usarla
donde se espera una distancia.


In [ ]:
print('similitud coseno   s(0,1)   = %.4f' % (1 - coseno(X[0], X[1])))
print('disimilitud coseno d(0,1)   = %.4f' % coseno(X[0], X[1]))
print('disimilitud coseno d(0,100) = %.4f' % coseno(X[0], X[100]))
print()
print('Verificamos la conversion s = 1/(1+d) del texto:')
d = euclidiana(X[0], X[100])
print('  d = %.4f  ->  s = %.4f' % (d, 1 / (1 + d)))

In [ ]:
# La misma matriz, con otra medida
R_cos = matriz_distancias(X, distancia=coseno)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, M, t in zip(axes, [R, R_cos], ['Euclidiana', 'Coseno']):
    im = ax.imshow(M, cmap='viridis')
    for corte in [50, 100]:
        ax.axhline(corte - 0.5, color='white', lw=1)
        ax.axvline(corte - 0.5, color='white', lw=1)
    ax.set_title(t)
    ax.set_xticks([25, 75, 125]); ax.set_xticklabels(iris.target_names, fontsize=8)
    ax.set_yticks([25, 75, 125]); ax.set_yticklabels(iris.target_names, fontsize=8)
    fig.colorbar(im, ax=ax)
plt.show()

Las dos matrices muestran la misma estructura de tres bloques, pero **no son la misma matriz**.
El coseno ignora la magnitud del vector: dos flores con la misma "forma" (misma proporción entre
pétalo y sépalo) pero tamaños distintos son idénticas para el coseno y distintas para la
euclidiana.

Elegir la medida es una decisión **del problema**, no del algoritmo. En documentos de texto el
coseno suele ser preferible, porque un documento largo y uno corto sobre el mismo tema tienen
frecuencias parecidas en proporción pero muy distintas en magnitud.

> **Ejercicio 5.1** Toma dos flores, multiplica una por 3 y calcula ambas distancias entre la
> original y la escalada. ¿Cuál cambia y cuál no? Explica el resultado a partir de las
> definiciones.


## 6. Relaciones sobre secuencias

En esta sección consideramos medidas que se aplican sobre secuencias de valores de características o vectores de características, por ejemplo, secuencias de valores de temperatura diarios, documentos de texto (secuencias de caracteres alfanuméricos) o secuencias de páginas web visitadas. Formalmente, dichas secuencias podrían verse como vectores de características, pero es más adecuado considerar explícitamente el carácter secuencial, el hecho de que cada elemento de la secuencia se refiera a la misma característica, y poder comparar secuencias de diferente longitud.

Usamos una función $\rho$ para comparar pares individuales de elementos de secuencia. Un ejemplo es la función de desigualdad binaria  que se usa en la distancia de Hamming (por lo que está se puede utilizar como una relación para secuencias de igual longitud).

Para calcular la relación entre secuencias de diferentes longitudes, se pueden agregar elementos neutrales como ceros o caracteres de espacio a la secuencia más corta. Dependiendo de la alineación de las subsecuencias, se pueden lograr distancias de Hamming más bajas anteponiendo o incluso insertando de manera óptima elementos neutrales. Esta es la idea detrás de la distancia de Levenshtein o distancia de edición que determina el número mínimo de operaciones de edición (insertar, eliminar o cambiar un elemento de secuencia) necesarias para transformar una secuencia en otra. Denotamos $L_{ij} (x, y)$ como la distancia de edición entre los primeros $i$ elementos de $x$ y los primeros $j$ elementos de $y$, y de forma recursiva se define la distancia de edición como:

$$L_{ij}=\begin{cases}
  i \text{ si } j=0\\
  j \text{ si } i=0\\
  \min\{L_{i-1,j}+1,\; L_{i,j-1}+1,\; L_{i-1,j-1}+\rho(x^{(i)},y^{(j)})\}
\end{cases}$$

En el tercer caso, $\rho$ vale 0 si los caracteres coinciden y 1 si no, de modo que una
sustitución cuesta 1 y una coincidencia no cuesta nada.


Los primeros dos casos consideran secuencias vacías y terminan la recursión (ya sea $x$ o $y$). En el tercer caso, los tres argumentos del operador mínimo corresponden a las tres operaciones de edición insertar, eliminar y cambiar. Para calcular $L_{ij}$ tenemos que calcular todo $L_{ij}, i = 1 ,\dots, p_x , j = 1,\dots, p_y$. La implementación recursiva es ineficiente porque recalcula los mismos subproblemas muchas veces; una solución con programación dinámica los calcula una sola vez y los guarda en una matriz:

In [ ]:
def levenshtein_distance(x, y):
    m = len(x) # numero de elementos en la secuencia x
    n = len(y) # numero de elementos en la secuencia y

    # Crear una matriz de (m+1) x (n+1) para almacenar los subproblemas
    # Hacemos m+1 y n+1 para agregar un vacio al inicio
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # Inicializar la primera fila y columna de la matriz
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)

    # Calcular la distancia de Levenshtein para los subproblemas restantes
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    # Devolver el resultado final y  la matriz resultante
    return dp[m][n], dp

Comparemos las secuencias *INFOTEC* e *INFORMACION*.


In [ ]:
x, y = "INFOTEC", "INFORMACION"
d, L = levenshtein_distance(x, y)
print(f"El numero necesario de ediciones para transformar {x} en {y} es: {d}")

La siguiente tabla muestra la matriz de distancias de edición $L$. Cada elemento se calcula
como el mínimo entre su vecino superior más uno, su vecino izquierdo más uno, y su vecino
diagonal superior izquierdo más la distancia entre los caracteres correspondientes. La distancia
entre ambas secuencias es el valor inferior derecho: en este caso **6**. Es decir, necesitamos al
menos seis operaciones de edición para convertir *INFOTEC* en *INFORMACION*, o viceversa.


In [ ]:
pd.DataFrame(L,columns=[" "]+list(y), index=[" "]+list(x))

Si calculamos el mínimo de cada columna podemos ver dónde se necesita una edición: el valor
aumenta exactamente en las columnas que obligan a editar.


In [ ]:
np.min(L, axis=0) # minimo de cada columna

El resultado es `[0 0 0 0 0 1 2 3 3 4 5 6]`. Leámoslo:

- Los primeros cinco valores son 0 porque *INFO* coincide en ambas cadenas (son cinco y no cuatro
  porque la primera columna corresponde a la cadena vacía).
- En la columna de la **R** el mínimo sube a 1: hay que sustituir la *T* de INFOTEC por una *R*.
- En **M** sube a 2 y en **A** a 3: dos inserciones para convertir INFOTEC en INFORMACION, o dos
  borrados en sentido inverso.
- En la segunda **C** se mantiene en 3, porque ese carácter sí existe en INFOTEC.
- En **I**, **O** y **N** sube a 4, 5 y 6: una inserción (o borrado) en cada caso.

> **Ejercicio 6.1** Calcula a mano la distancia entre *COCAS* y *HOLA*, verifica con el código
> y describe la secuencia concreta de ediciones.


## 7. El efecto de la escala

Ya tenemos una distancia y una matriz $R$. Una pregunta que debemos hacernos es la siguiente:
**¿en qué unidades están medidas las características?**

Volvamos al resumen estadístico de Iris.


In [ ]:
resumen = pd.DataFrame(X, columns=iris.feature_names).describe().loc[['min','max','std']]
resumen.loc['rango'] = resumen.loc['max'] - resumen.loc['min']
resumen

Los rangos no son iguales: la longitud del sépalo varía unos 3.6 cm, mientras que el ancho del
pétalo varía 2.4 cm. La diferencia es moderada porque las cuatro están en centímetros.

Ahora supongamos que quien tomó los datos midió el **sépalo en milímetros** y el pétalo en
centímetros. Es un descuido perfectamente realista, y no cambia la información: es la misma flor.


In [ ]:
X_mm = X.copy()
X_mm[:, 0] *= 10   # sepal length: cm -> mm
X_mm[:, 1] *= 10   # sepal width : cm -> mm

pd.DataFrame(X_mm, columns=iris.feature_names).describe().loc[['min','max','std']]

### 7.1 Qué le pasa con los objetos más cercanos

Buscamos, para una flor cualquiera, cuál es la flor más parecida a ella. Es literalmente k-NN
con $k=1$, y es también el `argmin` que usará k-centers para asignar cada punto a su centro.


In [ ]:
def vecino_mas_cercano(datos, i, distancia=euclidiana):
    """Indice del elemento mas cercano a datos[i] (sin contarse a si mismo)."""
    d = np.array([distancia(datos[i], datos[j]) for j in range(len(datos))])
    d[i] = np.inf          # que no se elija a si mismo
    return int(np.argmin(d))

for i in [0, 70, 120]:
    v_cm = vecino_mas_cercano(X, i)
    v_mm = vecino_mas_cercano(X_mm, i)
    print('flor %3d (%-10s)  vecino en cm: %3d (%-10s)   vecino en mm: %3d (%-10s)  %s'
          % (i, iris.target_names[Y[i]],
             v_cm, iris.target_names[Y[v_cm]],
             v_mm, iris.target_names[Y[v_mm]],
             '' if v_cm == v_mm else '<-- CAMBIO'))

In [ ]:
# Cuantas flores cambian de vecino mas cercano al cambiar las unidades?
vec_cm = [vecino_mas_cercano(X, i) for i in range(len(X))]
vec_mm = [vecino_mas_cercano(X_mm, i) for i in range(len(X))]
cambios = sum(a != b for a, b in zip(vec_cm, vec_mm))
print('%d de %d flores (%.0f%%) cambian de vecino mas cercano solo por el cambio de unidades'
      % (cambios, len(X), 100 * cambios / len(X)))

El problema es el siguiente:

> **La distancia euclidiana no es invariante al cambio de unidades.** Al multiplicar una
> característica por 10, su contribución a la distancia se multiplica por 100, porque entra al
> cuadrado. La característica con el rango más grande domina la distancia y las demás quedan
> prácticamente ignoradas.

No es un detalle de implementación: es una propiedad de la métrica. Y como k-centers, k-means y
k-NN **son** distancias más un `argmin`, el problema los afecta a los tres por igual.


### 7.2 La solución: llevar todas las características a una escala común

Hay dos transformaciones estándar:

**Estandarización** (z-score). Cada característica queda con media 0 y desviación estándar 1:

$$z^{(j)} = \frac{x^{(j)} - \mu_j}{\sigma_j}$$

**Normalización min-max.** Cada característica queda en el intervalo $[0,1]$:

$$x'^{(j)} = \frac{x^{(j)} - \min_j}{\max_j - \min_j}$$

La estandarización es más robusta ante valores atípicos; min-max es preferible cuando necesitas
un rango acotado. Ambas están en `sklearn.preprocessing`.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

Xz    = StandardScaler().fit_transform(X)     # a partir de los cm
Xz_mm = StandardScaler().fit_transform(X_mm)  # a partir de los mm

print('Tras estandarizar, cm y mm dan la MISMA matriz:', np.allclose(Xz, Xz_mm))
print()
vec_z    = [vecino_mas_cercano(Xz, i)    for i in range(len(X))]
vec_z_mm = [vecino_mas_cercano(Xz_mm, i) for i in range(len(X))]
print('flores que cambian de vecino tras estandarizar:',
      sum(a != b for a, b in zip(vec_z, vec_z_mm)))

Cero cambios. Estandarizar vuelve el resultado **independiente de las unidades**, que es
justo lo que queremos: la respuesta no debería depender de si alguien midió en centímetros o en
milímetros.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, datos, t in zip(axes, [X, X_mm, Xz],
                        ['Original (cm)', 'Sepalo en mm', 'Estandarizado']):
    M = matriz_distancias(datos)
    im = ax.imshow(M, cmap='viridis')
    for corte in [50, 100]:
        ax.axhline(corte - 0.5, color='white', lw=1)
        ax.axvline(corte - 0.5, color='white', lw=1)
    ax.set_title(t)
    ax.set_xticks([25, 75, 125]); ax.set_xticklabels(iris.target_names, fontsize=8)
    ax.set_yticks([25, 75, 125]); ax.set_yticklabels(iris.target_names, fontsize=8)
    fig.colorbar(im, ax=ax)
plt.show()

En la matriz del centro los bloques se desdibujan: al dominar el sépalo, que es la
característica que **peor** separa las especies, la estructura de grupos se degrada. La
estandarización la recupera.

### 7.3 Cuándo escalar y cuándo no

Escalar no siempre es lo correcto. La pregunta a hacerse es si las diferencias de rango entre
características son **información real** o un **artefacto de las unidades**.

| Situación | Escalar |
|---|---|
| Características en unidades distintas (cm, kg, pesos, años) | sí |
| Una característica con rango mucho mayor que las demás | sí, casi siempre |
| Píxeles de una imagen, todos en 0–255 | no, ya comparten escala |
| Vectores one-hot | no, ya son 0/1 |
| Sabes que una característica *debe* pesar más | no, o escala y luego pondera |

> **Ejercicio 7.1** Repite el experimento de la sección 7.1 con `MinMaxScaler` en lugar de
> `StandardScaler`. ¿Da los mismos vecinos? Si no, ¿cuál te parece más razonable para Iris y por qué?
>
> **Ejercicio 7.2** Repite el conteo de cambios usando `coseno` en lugar de `euclidiana`.
> ¿El coseno sufre el mismo problema? Cuidado: la respuesta no es la misma si escalas *todas* las
> características por el mismo factor que si escalas *una sola*.


## 8. Qué sigue: k-centers

Ya tenemos las tres piezas que necesita el primer algoritmo que evaluaremos:

1. Una matriz $X$ de datos numéricos, **con las características en una escala comparable**
   (notebook 01 + sección 7).
2. Una función de distancia $d(x,y)$, elegida según el problema (secciones 3 y 5).
3. La matriz de relaciones $R$ y el `argmin` sobre distancias (secciones 4 y 7.1).

El algoritmo de **k-centers** los combina así:

```
1. elegir un primer centro (por ejemplo, al azar)
2. repetir k-1 veces:
       para cada punto, calcular la distancia a su centro MAS CERCANO   <- argmin, seccion 7.1
       elegir como nuevo centro el punto cuya distancia sea MAYOR       <- argmax
3. asignar cada punto a su centro mas cercano                           <- argmin
```

Es decir: los centros se colocan lo más separados posible. En el siguiente notebook lo
implementarás sobre la plantilla `Clustering` de `src/SimpleClustering.py`, la misma clase que ya
contiene `euclidiana` y `coseno`.

Después veremos que **k-means** cambia solo el paso 2 (en vez de elegir el punto más lejano,
promedia los puntos de cada grupo) y que **k-NN** usa la misma maquinaria para clasificar en
lugar de agrupar. Los tres algoritmos son la misma idea con distintas decisiones.

## Resumen

| Concepto | Dónde se usa después |
|---|---|
| Axiomas de distancia | validar cualquier medida que propongas |
| Matriz $R$ de $n\times n$ | entrada de k-centers; motiva por qué k-means escala mejor |
| Euclidiana vs coseno | elegir la medida según el problema |
| Hamming | datos categóricos codificados con one-hot |
| Levenshtein | secuencias y texto de longitud variable |
| **Escalado** | **k-centers, k-means y k-NN, sin excepción** |
